In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("GPU not detected")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0


In [2]:
!pip install -q --upgrade torch torchvision torchaudio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 854.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 4.9 MB/s eta 0:00:00


In [3]:
import pandas as pd

df = pd.read_csv('/content/tickets.csv')

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (25921, 9)


,ticket_id,created_date,closed_date,agency,complaint_type,descriptor,borough,status,resolution_description
0,SR60000026,2025-01-01 00:17:14,2025-01-01 05:17:14,HPD,Heat/Hot Water,No Heat,MANHATTAN,Closed,The Department of Housing Preservation and Dev...
1,SR60000050,2025-01-01 00:29:24,2025-01-01 02:29:24,DOHMH,Rodent,Mouse Sighting,QUEENS,Closed,The Department of Health and Mental Hygiene in...
2,SR60000052,2025-01-01 00:34:48,NaN,DOHMH,Rodent,Rat Sighting,MANHATTAN,Pending,NaN
3,SR60000029,2025-01-01 00:37:17,2025-01-02 09:37:17,NYPD,Illegal Parking,Blocked Driveway,QUEENS,Closed,The Police Department responded to the complai...
4,SR60000017,2025-01-01 01:54:29,2025-01-03 12:54:29,HPD,Heat/Hot Water,No Heat,BRONX,Closed,The Department of Housing Preservation and Dev...


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25921 entries, 0 to 25920
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   ticket_id               25921 non-null  object
 1   created_date            25921 non-null  object
 2   closed_date             22760 non-null  object
 3   agency                  25921 non-null  object
 4   complaint_type          25921 non-null  object
 5   descriptor              25921 non-null  object
 6   borough                 25921 non-null  object
 7   status                  25921 non-null  object
 8   resolution_description  22760 non-null  object
dtypes: object(9)
memory usage: 1.8+ MB


In [5]:
df.describe()

,ticket_id,created_date,closed_date,agency,complaint_type,descriptor,borough,status,resolution_description
count,25921,25921,22760,25921,25921,25921,25921,25921,22760
unique,25921,25904,22750,6,7,20,5,4,16
top,SR60025891,2025-10-04 04:22:55,2025-12-15 21:57:50,NYPD,Noise - Residential,Loud Talking,BROOKLYN,Closed,The Police Department responded to the complai...
freq,1,2,2,16771,13384,4501,11988,22760,5886


In [6]:
df['complaint_type'].value_counts()

,count
complaint_type,
Noise - Residential,13384
Illegal Parking,3759
Street Condition,2136
Rodent,2040
Heat/Hot Water,1992
Sanitation Condition,1422
Water System,1188


In [7]:
df.groupby('complaint_type')['descriptor'].nunique()

,descriptor
complaint_type,
Heat/Hot Water,3
Illegal Parking,3
Noise - Residential,3
Rodent,3
Sanitation Condition,2
Street Condition,3
Water System,3


In [8]:
for complaint_type, group in df.groupby('complaint_type'):
    descriptors = sorted(group['descriptor'].dropna().unique())

    print(f"\n{complaint_type}")
    print(f"Number of descriptors: {len(descriptors)}")

    for descriptor in descriptors:
        print(f"  - {descriptor}")


Heat/Hot Water
Number of descriptors: 3
  - Inadequate Heat
  - No Heat
  - No Hot Water

Illegal Parking
Number of descriptors: 3
  - Blocked Driveway
  - Blocked Hydrant
  - Double Parked

Noise - Residential
Number of descriptors: 3
  - Banging/Pounding
  - Loud Music/Party
  - Loud Talking

Rodent
Number of descriptors: 3
  - Mouse Sighting
  - Rat Sighting
  - Signs of Rodents

Sanitation Condition
Number of descriptors: 2
  - Dirty Sidewalk
  - Overflowing Litter Basket

Street Condition
Number of descriptors: 3
  - Cave-In
  - Failed Street Repair
  - Pothole

Water System
Number of descriptors: 3
  - Catch Basin Clogged
  - Hydrant Leaking
  - Water Main Break


In [9]:
descriptor_counts = (
    df.groupby(['complaint_type', 'descriptor'])
      .size()
      .reset_index(name='count')
      .sort_values(['complaint_type', 'count'], ascending=[True, False])
)

descriptor_counts

,complaint_type,descriptor,count
0,Heat/Hot Water,Inadequate Heat,699
2,Heat/Hot Water,No Hot Water,684
1,Heat/Hot Water,No Heat,609
3,Illegal Parking,Blocked Driveway,1281
4,Illegal Parking,Blocked Hydrant,1258
5,Illegal Parking,Double Parked,1220
8,Noise - Residential,Loud Talking,4501
6,Noise - Residential,Banging/Pounding,4448
7,Noise - Residential,Loud Music/Party,4435
9,Rodent,Mouse Sighting,688


In [10]:
resolution_counts = (
    df.groupby('descriptor')['resolution_description']
      .nunique()
      .sort_values(ascending=False)
)

resolution_counts

,resolution_description
descriptor,
Blocked Driveway,3
Blocked Hydrant,3
Inadequate Heat,3
Double Parked,3
No Hot Water,3
No Heat,3
Catch Basin Clogged,2
Banging/Pounding,2
Failed Street Repair,2


In [11]:
for descriptor, group in df.groupby('descriptor'):
    resolutions = group['resolution_description'].dropna().unique()

    print(f"\n{'='*80}")
    print(f"DESCRIPTOR: {descriptor}")
    print(f"Unique resolutions: {len(resolutions)}")
    print(f"{'='*80}")

    for i, resolution in enumerate(resolutions, 1):
        print(f"\nResolution {i}:")
        print(resolution)


DESCRIPTOR: Banging/Pounding
Unique resolutions: 2

Resolution 1:
The Police Department responded to the complaint and determined that police action was not necessary.

Resolution 2:
The Police Department responded to the complaint and with the information available observed no evidence of the violation at that time.

DESCRIPTOR: Blocked Driveway
Unique resolutions: 3

Resolution 1:
The Police Department responded to the complaint and a report was prepared.

Resolution 2:
The Police Department responded and upon arrival those responsible were gone.

Resolution 3:
The Police Department responded to the complaint and took action to fix the condition.

DESCRIPTOR: Blocked Hydrant
Unique resolutions: 3

Resolution 1:
The Police Department responded to the complaint and a report was prepared.

Resolution 2:
The Police Department responded and upon arrival those responsible were gone.

Resolution 3:
The Police Department responded to the complaint and took action to fix the condition.

DESC

In [12]:
resolution_distribution = (
    df.groupby(['descriptor', 'resolution_description'])
      .size()
      .reset_index(name='count')
)

resolution_distribution['percentage'] = (
    resolution_distribution
    .groupby('descriptor')['count']
    .transform(lambda x: x / x.sum() * 100)
    .round(2)
)

resolution_distribution = resolution_distribution.sort_values(
    ['descriptor', 'count'],
    ascending=[True, False]
)

resolution_distribution

,descriptor,resolution_description,count,percentage
1,Banging/Pounding,The Police Department responded to the complai...,1963,50.03
0,Banging/Pounding,The Police Department responded to the complai...,1961,49.97
4,Blocked Driveway,The Police Department responded to the complai...,401,35.74
2,Blocked Driveway,The Police Department responded and upon arriv...,382,34.05
3,Blocked Driveway,The Police Department responded to the complai...,339,30.21
7,Blocked Hydrant,The Police Department responded to the complai...,384,34.88
5,Blocked Hydrant,The Police Department responded and upon arriv...,378,34.33
6,Blocked Hydrant,The Police Department responded to the complai...,339,30.79
9,Catch Basin Clogged,The Department of Environmental Protection inv...,169,51.21
8,Catch Basin Clogged,The Department of Environmental Protection det...,161,48.79


In [14]:
borough_resolution = (
    df.groupby(['descriptor', 'borough', 'resolution_description'])
      .size()
      .reset_index(name='count')
)

borough_resolution['percentage'] = (
    borough_resolution
    .groupby(['descriptor', 'borough'])['count']
    .transform(lambda x: x / x.sum() * 100)
    .round(2)
)

borough_resolution.sort_values(
    ['descriptor', 'borough', 'count'],
    ascending=[True, True, False]
)

,descriptor,borough,resolution_description,count,percentage
0,Banging/Pounding,BRONX,The Police Department responded to the complai...,194,53.30
1,Banging/Pounding,BRONX,The Police Department responded to the complai...,170,46.70
3,Banging/Pounding,BROOKLYN,The Police Department responded to the complai...,1211,50.58
2,Banging/Pounding,BROOKLYN,The Police Department responded to the complai...,1183,49.42
4,Banging/Pounding,MANHATTAN,The Police Department responded to the complai...,230,50.33
...,...,...,...,...,...
224,Water Main Break,MANHATTAN,The Department of Environmental Protection det...,33,42.31
227,Water Main Break,QUEENS,The Department of Environmental Protection inv...,46,60.53
226,Water Main Break,QUEENS,The Department of Environmental Protection det...,30,39.47
228,Water Main Break,STATEN ISLAND,The Department of Environmental Protection det...,9,64.29


In [15]:
# Convert date columns to datetime
df['created_date'] = pd.to_datetime(df['created_date'])
df['closed_date'] = pd.to_datetime(df['closed_date'])

# Calculate resolution time in hours
df['resolution_time_hours'] = (
    df['closed_date'] - df['created_date']
).dt.total_seconds() / 3600

# Check the result
df[['created_date', 'closed_date', 'resolution_time_hours']].head()

,created_date,closed_date,resolution_time_hours
0,2025-01-01 00:17:14,2025-01-01 05:17:14,5.0
1,2025-01-01 00:29:24,2025-01-01 02:29:24,2.0
2,2025-01-01 00:34:48,NaT,NaN
3,2025-01-01 00:37:17,2025-01-02 09:37:17,33.0
4,2025-01-01 01:54:29,2025-01-03 12:54:29,59.0


In [16]:
resolution_time_analysis = (
    df.dropna(subset=['resolution_description', 'resolution_time_hours'])
      .groupby('resolution_description')['resolution_time_hours']
      .agg(['count', 'mean', 'median', 'min', 'max'])
      .sort_values('median', ascending=False)
)

resolution_time_analysis

,count,mean,median,min,max
resolution_description,,,,,
The Police Department responded to the complaint and a report was prepared.,1049,41.215443,24.0,1.0,899.0
The Department of Health and Mental Hygiene inspected and found no rodent activity.,884,41.158371,23.5,1.0,1211.0
The Department of Transportation inspected the condition and completed the repair.,948,38.989451,23.0,1.0,634.0
The Department of Environmental Protection investigated this complaint and corrected the condition.,547,35.120658,23.0,1.0,480.0
The Department of Housing Preservation and Development inspected the following conditions. Violations were issued.,596,35.239933,22.0,1.0,453.0
The Police Department responded to the complaint and determined that police action was not necessary.,5881,37.107975,22.0,1.0,733.0
The Department of Environmental Protection determined that this condition was the responsibility of a private party.,503,37.856859,22.0,1.0,699.0
The Department of Health and Mental Hygiene inspected and found rodent activity. A notice of violation was issued.,875,37.250286,22.0,1.0,551.0
The Police Department responded to the complaint and with the information available observed no evidence of the violation at that time.,5886,36.538736,22.0,1.0,770.0
